# Zadanie 1, Część II – Text Embeddings (2 pkt) · PolEmo 2.0

Notebook odpowiada punktom z instrukcji (Text Embeddings, 2 pkt):

| Pkt zadania | Punkty | Sekcja w notebooku |
|---|---|---|
| 1. Osadzenia zdań, 2 różne modele (fastText / uśrednione Word2Vec / USE) | 0,5 | **Sekcja 1** |
| 2. t-SNE z odniesieniem do etykiet, wykresy interaktywne (hover: zdanie i anotacja) | 0,5 | **Sekcja 2** |
| 3. Analiza osadzeń i dyskusja (pytania pomocnicze 1–5) | 1,0 | **Sekcja 3** (3.1–3.5 materiał, 3.6 odpowiedzi) |

Kod - `src/text_embeddings.py`. W notebooku jest tylko uruchomienie i komentarz.
Etykiety do kolorowania: `gold` (PolEmo), `A1`, `A2` (niezależne anotacje obu iteracji) oraz `final`
(uzgodniona wersja po rozstrzygnięciu konfliktów przez superanotatora, z `annotation/iterN/adjudicated.csv`).

In [ ]:
import os, sys, warnings, logging, contextlib, io
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")
for _n in ("huggingface_hub", "tensorflow", "absl"):
    logging.getLogger(_n).setLevel(logging.ERROR)
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from src import text_embeddings as te

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
# Wykresy interaktywne pod komórkami: „plotly_mimetype” rysują JupyterLab, Notebook 7 i VS Code (także w niezaufanym
# notebooku), „notebook_connected” – klasyczny Notebook i eksport do HTML (plotly.js z CDN, potrzebny internet).
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Konfiguracja

In [ ]:
MODELS        = ["fasttext", "use"]
ITERATIONS    = [1, 2]                # próbki z obu iteracji (po 100 zdań)
INCLUDE_BASELINE = True               # TF-IDF + SVD jako punkt odniesienia (nie liczy się do 2 modeli)
FORCE         = False                 # True = przelicz osadzenia mimo cache

FASTTEXT_PATH = None                  # None = pobierze z Hugging Face (clarin-pl/fastText-kgr10); lub ścieżka do .bin
W2V_PATH      = None                  # ścieżka do modelu z https://dsmodels.nlp.ipipan.waw.pl (wymagana dla "w2v")
USE_BACKEND   = "auto"                # "tfhub" (oryginalny USE), "distiluse" (destylat mUSE) lub "auto"

## 1. Osadzenia zdań [0,5 pkt]

**Dane:** zdania z próbek obu iteracji anotacji (`annotation/iter1`, `annotation/iter2`), etykieta `gold` z PolEmo,
anotacje A1 i A2 z obu iteracji (niezależne, przed rozstrzygnięciem konfliktów przez superanotatora;
zdania bez etykiety dostają w kolorowaniu „brak”). Etykieta `final` to uzgodniona wersja po rozstrzygnięciu konfliktów przez superanotatora.

**Modele** (każdy daje jeden wektor na zdanie):
* **fastText (kgr10)** – średnia wektorów słów zdania; nieznane słowa mają wektor z n-gramów znakowych, więc
  nie ma problemu OOV (ważne przy artefaktach tokenizacji korpusu, np. „wymagał em”).
* **Word2Vec (IPI PAN)** – średnia wektorów słów; słowa spoza słownika są pomijane (raportujemy pokrycie).
* **Universal Sentence Encoder (multilingual)** – wektor całego zdania, bez uśredniania słów.
* **TF-IDF + SVD** – punkt odniesienia (worek słów), niczego nie pobiera.

In [ ]:
df = te.load_sentences(ITERATIONS)
print(len(df), "zdań;", "kolumny etykiet:", te.label_columns(df))
display(df.groupby("iter")["gold"].value_counts().unstack().assign(razem=lambda x: x.sum(axis=1)))
display(df[["sent_id", "iter", "text", "gold", "A1", "A2"]].head(5))

**Kontrola etykiet `final`.** Poniżej liczba zdań z rozstrzygnięciem i tabela: etykieta gold z PolEmo (wiersze) kontra
uzgodniona etykieta `final` (kolumny). Wartości poza przekątną to zdania, w których zespół uznał inną etykietę niż korpus.

In [ ]:
if "final" in te.label_columns(df):
    print("zdań z etykietą final:", int(df["final"].notna().sum()), "z", len(df))
    display(pd.crosstab(df["gold"], df["final"].fillna("brak"), rownames=["gold"], colnames=["final"], margins=True))
    ok = df["final"].notna()
    print(f"zgodność final z gold: {(df.loc[ok, 'final'] == df.loc[ok, 'gold']).mean():.1%}")
else:
    print("UWAGA: brak etykiet final – sprawdź annotation/iterN/adjudicated.csv (kolumny sent_id, final)")

**Jak czytać tabelę.** Wiersze to etykieta z korpusu PolEmo (`gold`), kolumny to etykieta po rozstrzygnięciu konfliktów (`final`), dla wszystkich 200 zdań z obu iteracji. Przekątna to zdania, w których korpus i zespół się zgadzają. Poza przekątną są zdania, którym zespół nadał inną etykietę niż korpus. Największa zmiana to `gold` AMB → `final` NEU, bo wg wytycznych AMB dotyczy sprzecznych ocen w jednym zdaniu, a nie ogólnego wydźwięku recenzji. To porównanie z korpusem, a nie miara zgodności anotatorów (tę liczy `python -m src.agreement`).

In [ ]:
names = MODELS + (["tfidf"] if INCLUDE_BASELINE else [])
embedders, E, failed = {}, {}, {}
for n in names:
    try:
        m = te.get_embedder(n, FASTTEXT_PATH, W2V_PATH, USE_BACKEND)
        E[n] = te.get_embeddings(m, df, force=FORCE)
        embedders[n] = m
        cov = getattr(m, "last_coverage", None)
        extra = f", pokrycie słownika: {cov.mean():.1%}" if cov is not None else ""
        print(f"OK  {n:9s} {E[n].shape}  {m.label}{extra}")
    except Exception as e:
        failed[n] = f"{type(e).__name__}: {e}"
        print(f"BŁĄD {n}: {failed[n]}")
real = [n for n in E if n != "tfidf"]
if len(real) < 2:
    warnings.warn(f"Zadanie wymaga 2 modeli, działają: {real}. Sprawdź błędy powyżej.")

### 1.3 Jak wyglądają osadzenia

Każde zdanie jest teraz **wektorem liczb**: fastText i TF-IDF dają 100 liczb, USE 512. Poniżej fragment macierzy
(pierwsze 5 zdań, pierwsze 8 wymiarów) oraz norma całego wektora. Pojedyncze wymiary nie mają interpretacji,
liczy się dopiero odległość między wektorami. Pełne osadzenia zapisują się do `results/text_embeddings/embeddings_<model>.csv`.

In [ ]:
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    print(f"macierz: {E[n].shape[0]} zdań × {E[n].shape[1]} wymiarów, dtype {E[n].dtype}, "
          f"średnia norma wektora: {np.linalg.norm(E[n], axis=1).mean():.2f}")
    display(te.embedding_preview(df, E[n], n_rows=5, n_dims=8))
    print("zapisano:", te.export_embeddings_csv(df, E[n], n))

**Podobieństwo kosinusowe wybranych zdań.** Po dwa zdania z każdej klasy gold (iteracja 1). Jasne pola = zdania bliskie
w danej przestrzeni, na przekątnej zawsze 1. w fastText wszystkie zdania są do siebie podobne,
w USE i TF-IDF większość par jest bliska zera.

In [ ]:
sel = (df[df["iter"] == df["iter"].min()].groupby("gold", sort=False).head(2)).sort_values("gold").index
for n in E:
    te.similarity_heatmap(df, E[n], sel, f"Podobieństwo kosinusowe – {embedders[n].label}").show()

## 2. Wizualizacja t-SNE z odniesieniem do etykiet [0,5 pkt]

Wykres jest interaktywny: **po najechaniu na punkt widać pełne zdanie oraz etykiety** (gold, A1, A2, final),
iterację i `sent_id`. Domyślnie punkty są kolorowane **uzgodnioną anotacją (`final`)**, a lista rozwijana nad wykresem przełącza kolorowanie między źródłami etykiet,
kliknięcie w legendę ukrywa klasę. Kółko = iteracja 1, romb = iteracja 2.
t-SNE: odległość kosinusowa, inicjalizacja PCA, `perplexity = min(30, (n−1)/3)`, stałe ziarno.
Wykresy zapisują się też jako HTML w `results/text_embeddings/`

In [ ]:
xy = {}
for n in E:
    xy[n] = te.tsne_2d(E[n])
    fig = te.make_tsne_figure(df, xy[n], f"t-SNE – {embedders[n].label}")
    fig.write_html(te.OUT_DIR / f"tsne_{n}.html", include_plotlyjs="cdn")
    fig.show()

**Jak czytać wykresy:** jeśli klasy wydźwięku tworzą oddzielne skupiska, osadzenie koduje sygnał przydatny
do klasyfikacji. Jeśli POZ i NEG są wymieszane, a skupiska mogą odpowiadać np. tematowi (hotel / lekarz /
szkoła / produkt) lub długości zdania, model koduje głównie temat.

## 3. Analiza osadzeń i dyskusja [1 pkt]

odpowiedzi w 3.6

### 3.1 Pyt. 1 – zdania uznane za podobne, a różniące się znaczeniem
Pary zdań o przeciwnym wydźwięku (POZ vs NEG według gold) z najwyższym podobieństwem kosinusowym.

In [ ]:
sims = {n: te.cosine_matrix(E[n]) for n in E}
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    display(te.similar_but_different(df, sims[n], k=8))

### 3.2 Pyt. 2 – zdania podobne znaczeniowo, a uznane za odległe (inne słowa)
Dwie metody. **(a)** Pary, które jeden model uznaje za bliskie, a drugi za odległe (rangi percentylowe wszystkich par) – to kandydaci, trzeba je ocenić ręcznie. **(b)** Kontrolowane pary: parafrazy (inne słowa, ten sam sens) vs. kontrola niepowiązana.

In [ ]:
if len(real) >= 2:
    a, b = real[0], real[1]
else:
    a, b = list(E)[:2]
if len(E) >= 2:
    hi_a, hi_b = te.cross_model_disagreement(df, sims[a], sims[b], a, b, k=8)
    print(f"Blisko według {a}, daleko według {b}:"); display(hi_a)
    print(f"Blisko według {b}, daleko według {a}:"); display(hi_b)

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    probes = te.probe_table(embedders)      # modele ładują się z dysku; komunikaty ukryte
display(probes)
print("Średnie cos. wg kategorii:")
display(probes.groupby("kategoria", sort=False)[list(E)].mean().round(3))

Zdania w tabeli powyżej nie pochodzą z korpusu. to zestaw kontrolny napisany ręcznie (`PROBE_PAIRS` w `src/text_embeddings.py`), zrobiony po to, żeby sprawdzić zjawiska z pytań 1–3, które w 200 losowych zdaniach trudno znaleźć. Kolumna `kategoria` mówi, jaka jest relacja między a i b: `parafraza` (ten sam sens, inne słowa), `przeciwny sens, podobne słowa` („smaczne” / „niesmaczne”), `przeciwny sens, te same słowa` (te same słowa w innej kolejności), `ten sam sens, różna długość` oraz `kontrola: niepowiązane` (linia bazowa: ile wynosi cosinus „z niczego”). Liczby w kolumnach modeli to cosinus między wektorami obu zdań. Dla 12 par wnioski są ilustracją, a nie statystyką; dane z korpusu są w 3.1 i 3.2(a).

### 3.3 Pyt. 3 – różna długość, podobny sens
Kategoria „ten sam sens, różna długość” w tabeli powyżej (porównaj z parafrazami i z kontrolą niepowiązaną).

In [ ]:
len_pairs = probes[probes["kategoria"].str.contains("różna długość")].copy()
len_pairs["dł_a"] = len_pairs["zdanie_a"].str.split().str.len()
len_pairs["dł_b"] = len_pairs["zdanie_b"].str.split().str.len()
display(len_pairs[["zdanie_a", "zdanie_b", "dł_a", "dł_b"] + list(E)])

### 3.4 Pyt. 4 – długość zdania a wartości podobieństwa i „wyrazistość” wektora
Przedziały długości (kwantyle liczby tokenów): norma surowego wektora (dla uśrednianych wektorów słów to miara siły sygnału; dla USE norma jest zwykle ≈ stała, więc nie ma interpretacji), cosinus do centroidu korpusu i średnie podobieństwo do pozostałych zdań.

In [ ]:
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    display(te.length_stats(df, E[n]))

fig = go.Figure()
for n in E:
    S = sims[n].copy(); np.fill_diagonal(S, np.nan)
    fig.add_trace(go.Scatter(x=df["n_tokens"], y=np.nanmean(S, axis=1), mode="markers", name=n,
                             opacity=0.6, text=df["text"], hovertemplate="%{text}<br>tokenów: %{x}, śr. cos: %{y:.3f}<extra>"+n+"</extra>"))
fig.update_layout(title="Długość zdania vs średnie podobieństwo do pozostałych zdań", template="plotly_white",
                  xaxis_title="liczba tokenów", yaxis_title="średni cosinus do innych zdań", height=520)
fig.show()

Zdania podzielono na 3 grupy wg liczby tokenów, tak aby w grupach było podobnie dużo zdań.

* `n` – liczba zdań w grupie.
* `norma_wektora` – średnia długość surowego wektora zdania. W fastText wektor zdania to średnia wektorów słów, więc w dłuższych zdaniach wektory częściowo się znoszą i norma maleje. W USE norma jest stała (1), więc ta kolumna nic nie mówi.
* `cos_do_centroidu` – podobieństwo zdania do środka (średniej) wszystkich zdań. Im wyżej, tym zdanie jest „bardziej przeciętne”.
* `srednie_cos_do_innych` – średnie podobieństwo zdania do wszystkich pozostałych.
* Ostatni wiersz – współczynnik Spearmana między liczbą tokenów a średnim cosinusem. Dodatni znak = im dłuższe zdanie, tym bardziej podobne do reszty.

Wniosek: w fastText długie zdania zbliżają się do średniego wektora, więc wysokie cosinusy mogą wynikać z długości, a nie z podobnego sensu.

### 3.5 Pyt. 5 – czy osadzenia nadają się do zadania (klasyfikacja wydźwięku zdania)?
Regresja logistyczna (walidacja krzyżowa warstwowa) na samych osadzeniach, czystość k najbliższych sąsiadów względem poziomu losowego i silhouette. **Punkt odniesienia dla accuracy to `baza_wiekszosciowa`** (udział najliczniejszej klasy, czyli wynik klasyfikatora zawsze wybierającego tę klasę); `losowy_wg_rozkladu` to losowanie zgodne z rozkładem klas, a `acc_minus_baza` mówi, o ile model go przekracza. Etykiety (gold, A1, A2, final) są liczone na zdaniach, które je mają. `final` to uzgodnione etykiety, więc to ją traktujemy jako główny punkt odniesienia; gold PolEmo i A1/A2 służą do porównania. Przy ~200 zdaniach wyniki są orientacyjne, nie ma tu przedziałów ufności.

In [ ]:
rows = []
for n in E:
    for col in te.label_columns(df):
        r = te.evaluate_embeddings(E[n], df[col]); r.update(model=n, etykiety=col); rows.append(r)
res = pd.DataFrame(rows).set_index(["model", "etykiety"])
display(res)

Klasyfikator (regresja logistyczna) przewiduje wydźwięk zdania z samego wektora zdania, czyli sprawdzamy jakość osadzeń, a nie samego klasyfikatora. Wiersze to model osadzeń (`fasttext`, `use`, `tfidf` jako punkt odniesienia) × zestaw etykiet, które traktujemy jako prawdę (`final`, `gold`, `A1`, `A2`). Wynik jest z 5-krotnej walidacji krzyżowej, więc klasyfikator nie ocenia zdań, na których się uczył.

* `accuracy` – odsetek poprawnych przewidywań. Porównuj ją z `baza_wiekszosciowa` (wynik klasyfikatora, który zawsze wybiera najliczniejszą klasę); `acc_minus_baza` to różnica. Przy `final` i A1/A2 klasa NEU jest najliczniejsza, więc same accuracy bywa mylące.
* `macro_F1` – średnia F1 po klasach, każda klasa liczy się tak samo, więc ignorowanie rzadkich klas (AMB) mocno obniża wynik.
* `kNN10_czystosc` – średni odsetek 10 najbliższych sąsiadów zdania (wg cosinusa) z tą samą etykietą. Nie używa klasyfikatora, mierzy samą geometrię osadzenia. Odnosi się do `losowy_wg_rozkladu` (losowanie zgodne z rozkładem klas).
* `silhouette_cos` – zwykle miara klasteryzacji, ale działa dla dowolnego podziału na grupy, także dla etykiet. Pokazuje, czy zdania jednej klasy leżą bliżej siebie niż zdań innych klas (od −1 do 1). Wartości ≤ 0 to brak oddzielnych skupisk, zgodnie z t-SNE.
* Różnice między `gold` a `final`/A1/A2: `gold` jest zbalansowane (po 50 zdań na klasę), a w `final` i A1/A2 dominuje NEU, więc accuracy jest wyższa głównie dlatego, że łatwiej trafić w NEU.

Macierz pomyłek (wiersze = uzgodniona etykieta `final`, kolumny = predykcja klasyfikatora na samych
osadzeniach) i precyzja / czułość / F1 dla każdej klasy. Pokazuje, czy problem leży w klasach „środkowych” (NEU, AMB), czy też
osadzenia mylą nawet POZ z NEG.

In [ ]:
main_col = "final" if "final" in te.label_columns(df) else "gold"
print("etykiety:", main_col)
for n in E:
    cm, rep = te.confusion_report(E[n], df[main_col])
    print(f"\n=== {embedders[n].label} ===")
    display(cm); display(rep)

Wiersz to prawdziwa etykieta (`final`), kolumna to to, co przewidział klasyfikator. Liczby na przekątnej to trafienia. Jeśli prawie wszystko ląduje w kolumnie NEU, klasyfikator „ucieka” do najliczniejszej klasy. Zwróć uwagę na `recall` NEG i AMB: bliskie 0 oznacza, że osadzenia prawie nie odróżniają tych klas od NEU.

### 3.6 Odpowiedzi na pytania pomocnicze (do uzupełnienia po uruchomieniu)

Każda odpowiedź ma opierać się na konkretnych przykładach i liczbach z sekcji 3.1–3.5.

**1. Czy są zdania, które model uznał za podobne, ale w rzeczywistości różnią się znaczeniem?**
_Odpowiedź:_ …

**2. Czy są zdania semantycznie podobne, ale model uznał je za odległe, bo użyto innych słów?**
_Odpowiedź:_ … 

**3. Czy zdania o różnej długości, ale podobnym sensie, są traktowane przez model podobnie?**
_Odpowiedź:_ … 

**4. Czy zauważasz różnicę w wartościach podobieństwa między krótkimi a długimi zdaniami? Czy długość wpływa na „wyrazistość” wektora?**
_Odpowiedź:_ … 

**5. Czy osadzenia zdań nadają się do wybranego zadania (anotacja wydźwięku)?**
_Odpowiedź:_ … 